# 动量表达改进：固定训练与完整Tiered回测

仅将重叠累计动量改为既定非重叠对数区间；其他信息、训练配置、约束、控制器与成本不变。固定三seed均值，不调参，不挑最好seed。历史已复用，不能称新鲜样本外。

In [1]:
from pathlib import Path
import os,json
import qlib
from src.workflow.allocator_representation import run
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/disjoint_momentum_training_20261014';plan=json.loads((card/'plan.json').read_text());qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
summary=run(root=root,card=card,plan=plan)
print({k:summary[k] for k in ['decision','new_fits','new_native_cases','gates']})


In [1]:
from pathlib import Path
import json,hashlib
import qlib
from qlib.workflow import R
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd();card=root/'research/disjoint_momentum_training_20261014';s=json.loads((card/'summary.json').read_text());qlib.init(provider_uri=s['provider_uri'],region='us',kernels=1)
with R.uri_context(s['recorder_uri']):
 rec=R.get_recorder(recorder_id=s['recorder_id'],experiment_name=card.name);table=rec.load_object('analysis/comparison');training=rec.load_object('analysis/training')
 candidate=table.loc[table.family.eq('candidate')].set_index('slippage_bps');anchor=table.loc[table.family.eq('current_anchor')].set_index('slippage_bps');s['gates']['beat_original_all_costs']=bool(candidate.terminal_wealth.gt(anchor.terminal_wealth).all());s['route_status']='closed_fixed_representation_no_window_or_seed_search';s['training_audit']=[{k:v for k,v in row.items() if k in ['seed','training_rows','auxiliary_rows','epochs','checkpoint_policy','first_training_signal','last_training_signal','last_mature_label_at','primary_dimensions']} for row in training]
 with R.start(experiment_name=card.name,recorder_id=s['recorder_id'],resume=True):
  rec=R.get_recorder();rec.save_objects(artifact_path='analysis',summary=s,original_anchor_gate={'formula':'candidate terminal wealth > original anchor terminal wealth separately for every fixed cost','passed':s['gates']['beat_original_all_costs'],'purpose':'Explicitly report the gate already listed in frozen plan; no refits or account replays'})
  artifacts=Path(s['recorder_local_dir'])/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest(),'bytes':p.stat().st_size} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');s['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=s['experiment_id']);atomic_json(s,card/'summary.json');print(s['training_audit'])


# 动量分段训练与完整净费回测：未通过，固定路线关闭

实际完成3个候选模型训练、3套新的Qlib原生完整账户回测。匹配基线的3个模型经逐值预测复核后复用，9套既有原生账户包括3个匹配基线、3个原模拟盘模型重建账户、3个SOXX买入持有对照。没有挑选最优seed；固定71/172/273等权平均。

每个候选使用3014个成熟主训练样本、20741个辅助样本，300轮；最早/最晚训练信号2011-01-03 / 2022-12-21，最后成熟标签2022-12-30 09:31:00-05:00。采用原固定最终轮检查点，不根据2023+回测选择epoch。训练/推断通过Qlib DatasetH，原生价格和回测来源固定。

唯一变化是primary输入中SOXL/SOXX各5个累计动量改为1–3、4–5、6–10、11–20、21–60区间的对数收益。可逆重构检查通过，维数、共同编码器、标签、损失、初始化、训练范围、超参数、约束和完整执行控制器保持配对。共同编码器内的其他动量输入仍保留，本轮不同时做去重或目标函数修改。不是新增信息源。

10万美元初始本金，IBKR Pro Tiered沿用已冻结费用投影，0/2/5bp分别独立持仓、风控和月度计费状态；现金起点与基准一致。944个可执行区间，自2023-01-03信号起至2026-10-09结算。费用是既定研究投影，不是历史实际账单。

|模型|滑点|累计净收益|CAGR|收盘最大回撤|盘中监控最大回撤|相对SOXX财富变化|
|---|---|---|---|---|---|---|
|分段动量候选|0bp|+242.04%|38.86%|-23.99%|-24.89%|-29.87%|
|匹配基线|0bp|+280.24%|42.84%|-25.93%|-26.20%|-22.04%|
|原模拟盘模型重建|0bp|+228.99%|37.42%|-28.11%|-29.50%|-32.55%|
|SOXX|0bp|+387.75%|52.66%|-42.87%|-43.38%|+0.00%|
|分段动量候选|2bp|+240.54%|38.69%|-24.04%|-24.93%|-30.18%|
|匹配基线|2bp|+269.16%|41.72%|-24.40%|-25.77%|-24.31%|
|原模拟盘模型重建|2bp|+279.67%|42.78%|-24.17%|-27.07%|-22.16%|
|SOXX|2bp|+387.73%|52.65%|-42.87%|-43.39%|+0.00%|
|分段动量候选|5bp|+233.26%|37.90%|-24.15%|-24.99%|-31.67%|
|匹配基线|5bp|+266.08%|41.40%|-24.46%|-25.83%|-24.94%|
|原模拟盘模型重建|5bp|+41.12%|9.63%|-31.38%|-31.42%|-71.06%|
|SOXX|5bp|+387.70%|52.65%|-42.88%|-43.39%|+0.00%|

相对财富变化是候选增长倍数除以SOXX增长倍数减1，不是收益百分点。2bp候选落后匹配基线约28.62个累计收益百分点，5bp落后约32.82个百分点。虽然5bp明显好于原模拟盘单模型，但匹配基线本来就更好，不能把那段提升归功于动量分段。2bp候选也未超过原模拟盘模型。

|年度（按信号年归因）|2bp候选净收益|2bp相对SOXX财富变化|5bp候选净收益|5bp相对SOXX财富变化|
|---|---|---|---|---|
|2023|+41.90%|-9.39%|+41.19%|-9.83%|
|2024|+28.33%|+6.08%|+27.74%|+5.59%|
|2025|+33.84%|-8.83%|+33.20%|-9.27%|
|2026|+39.73%|-20.33%|+38.73%|-20.90%|

候选最大回撤约24%（盘中监控约25%），满足30%约束，CVaR也优于SOXX；但净SOXX超额、匹配基线、年度/126日滚动稳定性与区块置信门均未通过。2bp实际平均现金由基线46.43%升至47.77%，没有解决现金偏高的问题；不能仅凭这个均值把全部收益差归因于现金。

这是一次真实训练与完整策略回测，但2023+历史已被反复查看，缺独立开发选择及合格前向证据，不是新鲜样本外。原模拟盘比较项是同现金起点、同成本引擎的历史重建，不是声称实盘已获得这些收益。

决定：not_promoted_fixed_representation_failed。输入共线性下降未转化为净收益改善，关闭此固定表达路线，不改窗口、损失、epoch或挑seed挽救。现役模型与production release未更新，稳定净超额目标仍未实现。后续工作继续围绕历史训练与完整回测，不等待实时窗口；新实验必须另有明确机制并与强配对基线比较。

验证：既有特征合同2项通过；执行中验证初始参数相同、训练域一致、未变化特征及scaler一致、控制预测与保存结果逐值一致、完整账户共同日历一致。NAS冷读回见readback_verification.json。

[完整比较](backtest_benchmark_comparison.csv) · [年度结果](annual_metrics.csv) · [区块区间](uncertainty.csv) · [固定计划](plan.json)

Recorder `5925e4ebbde6477e8fd4e99771ef72dd`。
